#                                  **தமிழ்நாடு திட்ட வழிகாட்டி**

# Install Required Libraries


In [ ]:
!pip -q install groq sentence-transformers faiss-cpu pypdf

# Setup (API Key and Model Selection)

In [ ]:
import os
import glob
import numpy as np
import faiss
from groq import Groq
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# --- API key: from Colab Secrets, otherwise ask you to type it ---
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Paste your Groq API key: ")

client = Groq()

# --- Pick an LLM that is actually available on your account ---
PREFERRED_MODELS = [
    "llama-3.3-70b-versatile",    # best Tamil quality
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "llama-3.1-8b-instant",
]
try:
    available = [m.id for m in client.models.list().data]
except Exception as e:
    raise SystemExit(f"Could not connect to Groq. Check your API key in Colab Secrets.\nDetails: {e}")

skip_words = ["whisper", "tts", "guard", "orpheus"]
chat_models = [m for m in available if not any(w in m.lower() for w in skip_words)]
LLM_MODEL = next((m for m in PREFERRED_MODELS if m in available), chat_models[0])

# Free local embedding model (e5 supports Tamil). Groq has no embeddings API.
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

print("Setup done")
print("LLM in use:", LLM_MODEL)

# Knowledge Base (Sample TN Scheme Data)

In [ ]:
# Tamil Nadu welfare schemes - verified against news reports / official portals
# Last verification pass: 11 Oct 2026
# NOTE: Tamil Nadu changed government in May 2026 (TVK, CM C. Joseph Vijay). Several schemes were
# renamed or expanded. Both old and new names are kept in the text so either name retrieves the right document.

documents = [
    {
        "title": "Kalaignar Magalir Urimai Thogai (now Madhippumigu Magalir Thittam)",
        "source": "https://kmut.tn.gov.in (official portal); news: https://newstodaynet.com/2026/06/15/mut-tn-govt-credits-%E2%82%B91000-to-1-31cr-women-for-this-month/",
        "last_verified": "2026-10-11",
        "text": (
            "Kalaignar Magalir Urimai Thogai (KMUT), also called the Magalir Urimai Thogai or the Rs 1,000 scheme, "
            "is a monthly cash assistance scheme of Rs 1,000 for the woman head of an eligible family in Tamil Nadu. "
            "It was announced in March 2023 and officially launched on 15 September 2023. The money is credited "
            "directly to the beneficiary's bank account through Direct Benefit Transfer (DBT); under the earlier "
            "DMK government it was paid on the 15th of every month. About 1.31 crore women receive it. "
            "Eligibility conditions: the applicant must be a woman who is the head of the family on a valid "
            "ration card, aged 21 or above, a resident of Tamil Nadu, with annual family income below Rs 2.5 lakh. "
            "Only one woman per ration card is eligible, and families with income tax payers are excluded. "
            "The original 2023 guidelines also set limits of less than 5 acres of wet land or 10 acres of dry land "
            "and household electricity use below 3,600 units per year. "
            "Applications were first taken in special registration camps (24 July to 16 August 2023) at ration shops; "
            "applications, status checks and grievances are now handled through the portal kmut.tn.gov.in and "
            "e-Sevai centres, and the helpline is 044-25619208. "
            "Status in 2026: after the TVK government took office in May 2026, the scheme appears in news reports "
            "under the name Madhippumigu Magalir Thittam, and the Rs 1,000 monthly payment continues. The TVK "
            "election manifesto promised Rs 2,500 per month, but as of September 2026 reports say the amount has "
            "not been raised and the government needs time to restructure the scheme. Treat any claim of a "
            "Rs 2,500 payment as unconfirmed until an official government order is published."
        ),
    },
    {
        "title": "Pudhumai Penn Scheme (Moovalur Ramamirtham Ammaiyar Higher Education Assurance Scheme) and Tamil Pudhalvan",
        "source": "https://tils.tn.gov.in/schemes ; https://www.tribuneindia.com/news/india/tamil-nadu-cm-launches-rs-1000-monthly-assistance-scheme-for-boys/amp",
        "last_verified": "2026-10-11",
        "text": (
            "Pudhumai Penn, officially the Moovalur Ramamirtham Ammaiyar Higher Education Assurance Scheme, was "
            "launched on 5 September 2022. It gives Rs 1,000 per month to girl students who studied in Classes 6 to 12 "
            "in government or government-aided schools in Tamil Nadu and are now pursuing higher education such as a "
            "degree, diploma, ITI, vocational or paramedical course at a recognised institution in the state. "
            "The amount is transferred directly to the student's bank account for the duration of the course. "
            "Distance education and open university students are not covered. Students apply through their "
            "college with a school study certificate, bank passbook, Aadhaar and admission proof. "
            "The scheme aims to reduce dropouts among girls after school. "
            "Its counterpart for boys, Tamil Pudhalvan, was launched on 9 August 2024 and also pays Rs 1,000 per month "
            "to boys who studied Classes 6 to 12 in government schools (or in Tamil medium in government-aided "
            "schools). Both schemes have been extended to transgender and intersex persons, who need a Tamil Nadu "
            "Transgender Welfare Board identity card and are exempt from the Tamil-medium requirement. "
            "In 2025-26 the assistance was expanded to more college students. "
            "Note: it was not confirmed whether the 2026 government has renamed these schemes."
        ),
    },
    {
        "title": "Vidiyal Payanam (Free Bus Travel for Women) - expanded as Vettri Payanam",
        "source": "https://www.dtnext.in/news/tamilnadu/cm-vijay-rides-76b-to-kick-off-vettri-payanam-20 ; https://www.gktoday.in/tamil-nadu-launches-vetri-payanam-thittam-scheme/",
        "last_verified": "2026-10-11",
        "text": (
            "Vidiyal Payanam is the zero-fare bus travel scheme launched in May 2021 by the Tamil Nadu government. "
            "Under it, women (and transgender persons) can travel free in Ordinary Town buses run by the seven "
            "state transport undertakings, and in Ordinary Ghat buses on routes up to 40 km. It covered about "
            "7,537 Ordinary Town buses and 173 Ordinary Ghat buses, and an average of 71.7 lakh women travelled "
            "free each day in 2026-27. There is no income limit, caste restriction or cap on journeys per day, and "
            "passengers get a zero-fare ticket. "
            "On 2 October 2026, Chief Minister C. Joseph Vijay launched the expanded Vettri Payanam scheme at "
            "Koyambedu, Chennai. It extends free travel for women and transgender persons to Express, Limited Stop "
            "Service (LSS) and Deluxe government buses in urban and city areas, and to Mofussil Ordinary buses and "
            "Ghat Ordinary buses on hill routes longer than 40 km. About 12,692 of the 18,413 buses of the seven "
            "state transport undertakings (roughly 69 percent) are covered, and about 85 lakh women are expected to "
            "benefit daily. Private buses are not included. "
            "The scheme reduces daily travel cost for working women and students. "
            "Note: free travel for differently abled persons and their escorts is a separate, older concession "
            "and was not confirmed in the sources checked for this document."
        ),
    },
    {
        "title": "Chief Minister's Breakfast Scheme (now Perunthalaivar Kamarajar Breakfast Scheme)",
        "source": "https://gulfnews.com/world/asia/india/tamil-nadu-expands-free-breakfast-scheme-to-15-million-more-students-1.500683583 ; https://thesouthfirst.com/tamilnadu/tamil-nadu-expands-breakfast-scheme-to-government-aided-schools/",
        "last_verified": "2026-10-11",
        "text": (
            "The Chief Minister's Breakfast Scheme was launched on 15 September 2022 in Madurai to give a free, "
            "hot, nutritious breakfast to schoolchildren, because many children came to school on an empty stomach. "
            "It started with 1.14 lakh students of Classes 1 to 5 in 1,545 government primary schools, was extended "
            "in August 2023 to all government primary schools (about 31,000 schools), and in July 2024 to "
            "government-aided primary schools, bringing about 21.87 lakh students under it. In 2025 it was "
            "extended to another 2,430 government schools. The meal is served in school before classes begin "
            "(for example pongal, sambar, upma, kesari). Officials reported that attendance rose after the scheme. "
            "After the TVK government came to power, the scheme was renamed the Perunthalaivar Kamarajar Breakfast "
            "Scheme. On 22 September 2026 Chief Minister C. Joseph Vijay expanded it to Classes 6 to 8, adding about "
            "15.3 lakh students in 15,454 government and government-aided schools. It now covers about 34.6 lakh "
            "students from Classes 1 to 8 in 44,984 schools. The scheme aims to improve attendance, nutrition and "
            "learning outcomes of children."
        ),
    },
    {
        "title": "Chief Minister's Comprehensive Health Insurance Scheme (CMCHIS)",
        "source": "https://www.cmchistn.com ; https://www.dtnext.in/explainer/how-to-apply-for-tamil-nadu-cms-comprehensive-health-insurance-scheme-eligibility-documents-and-process ; https://www.outlookmoney.com/personal-finance/tamil-nadu-health-scheme-cover-raised-fivefold-to-rs-25-lakh-what-beneficiaries-should-know",
        "last_verified": "2026-10-11",
        "text": (
            "The Chief Minister's Comprehensive Health Insurance Scheme (CMCHIS), earlier known as the Kalaignar "
            "Kaappittu Thittam and merged with Ayushman Bharat (PMJAY-CMCHIS), has provided cashless treatment at "
            "empanelled government and private hospitals since 2009. Until recently the cover was Rs 5 lakh per "
            "family per year; in 2026 the Tamil Nadu government announced that the annual cover is being raised "
            "to Rs 25 lakh. The date from which the higher limit becomes operational, and whether new cards or "
            "verification are needed, had not been confirmed in the sources checked. "
            "Eligibility: the applicant must be a resident of Tamil Nadu and the family's annual income should not "
            "exceed Rs 1.20 lakh, shown by an income certificate from the VAO or revenue authorities. Sri Lankan "
            "Tamil refugees in camps can enrol without an income certificate. Applicants submit the family "
            "ration card, Aadhaar and income certificate at a CMCHIS registration centre, complete Aadhaar and "
            "biometric verification and receive a smart card. "
            "The scheme covers about 1,090 procedures, 8 follow-up procedures and 52 diagnostic procedures, "
            "including surgeries, cardiac care, oncology, maternity care and therapies, across roughly 800 "
            "government and 900 private empanelled hospitals. Treatment is covered only for procedures and "
            "hospitals listed under the scheme."
        ),
    },
    {
        "title": "Naan Mudhalvan Skill Development",
        "source": "https://naanmudhalvan.tn.gov.in/about ; https://www.dtnext.in/amp/story/news/tamilnadu/courses-under-naan-mudhalvan-scheme-to-get-a-makeover-811431",
        "last_verified": "2026-10-11",
        "text": (
            "Naan Mudhalvan is a skill development initiative of the Government of Tamil Nadu, launched by the "
            "Chief Minister on 1 March 2022 and run by the Tamil Nadu Skill Development Corporation (TNSDC). "
            "It offers free, employment-linked, industry-aligned training to college students and youth, generally "
            "aged 18 to 35, in areas such as IT, engineering, soft skills and emerging technologies, delivered with "
            "industry and training partners, with career guidance and skill certifications to improve "
            "employability. The initial target was to train more than 10 lakh youth, and by August 2024 the "
            "government reported about 28 lakh students trained. Its verticals include courses for engineering and "
            "polytechnic students, competitive-exam coaching (started in 2023, including UPSC support) and the "
            "Naan Mudhalvan Olympiad for Classes 6 to 8. "
            "Students and youth register through the Naan Mudhalvan online portal (naanmudhalvan.tn.gov.in), "
            "which was launched on 29 August 2022."
        ),
    },
]
print(f"Loaded {len(documents)} documents")

# Tamil Scheme Names (Glossary)

In [ ]:
TAMIL_NAMES = {
    "Kalaignar Magalir Urimai Thogai (now Madhippumigu Magalir Thittam)":
        "கலைஞர் மகளிர் உரிமைத் தொகை (மதிப்புமிகு மகளிர் திட்டம்)",
    "Pudhumai Penn Scheme (Moovalur Ramamirtham Ammaiyar Higher Education Assurance Scheme) and Tamil Pudhalvan":
        "புதுமைப் பெண் திட்டம் மற்றும் தமிழ்ப் புதல்வன் திட்டம்",
    "Vidiyal Payanam (Free Bus Travel for Women) - expanded as Vettri Payanam":
        "விடியல் பயணம் (வெற்றிப் பயணம்) - பெண்களுக்கு இலவசப் பேருந்துப் பயணம்",
    "Chief Minister's Breakfast Scheme (now Perunthalaivar Kamarajar Breakfast Scheme)":
        "முதலமைச்சரின் காலை உணவுத் திட்டம் (பெருந்தலைவர் காமராஜர் காலை உணவுத் திட்டம்)",
    "Chief Minister's Comprehensive Health Insurance Scheme (CMCHIS)":
        "முதலமைச்சரின் விரிவான மருத்துவக் காப்பீட்டுத் திட்டம்",
    "Naan Mudhalvan Skill Development":
        "நான் முதல்வன் திறன் மேம்பாட்டுத் திட்டம்",
}

for d in documents:
    d["tamil_name"] = TAMIL_NAMES.get(d["title"], "")

missing = [d["title"] for d in documents if not d["tamil_name"]]
if missing:
    print("WARNING: no Tamil name for:", missing)

GLOSSARY = "\n".join(f"- {d['title']} = {d['tamil_name']}" for d in documents if d["tamil_name"])
print(GLOSSARY)


# Chunking (Split Documents into Small Pieces)

In [ ]:
def chunk_text(text, chunk_size=120, overlap=20):
    words = text.split()
    step = chunk_size - overlap
    chunks, start = [], 0
    while True:
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
        start += step
    return chunks

def build_chunks(docs, chunk_size=120, overlap=20):
    result = []
    for doc in docs:
        header = doc["title"]
        if doc.get("tamil_name"):
            header += f" / {doc['tamil_name']}"
        for piece in chunk_text(doc["text"], chunk_size, overlap):
            result.append({
                "text": f"{header}: {piece}",
                "title": doc["title"],
                "source": doc["source"],
            })
    return result

chunks = build_chunks(documents)
print(f"{len(documents)} documents -> {len(chunks)} chunks")
print("\nExample chunk:\n", chunks[0]["text"])


# Embeddings and Vector Index (FAISS)

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)

def build_index(chunk_list):
    texts = ["passage: " + c["text"] for c in chunk_list]   # e5 needs this prefix
    vectors = embedder.encode(texts, normalize_embeddings=True).astype("float32")
    idx = faiss.IndexFlatIP(vectors.shape[1])
    idx.add(vectors)
    return idx, vectors

index, chunk_vectors = build_index(chunks)
print("Vector shape:", chunk_vectors.shape, "(chunks x dimensions)")
print("Chunks in index:", index.ntotal)


# Retrieval (Find Relevant Chunks)

In [ ]:
def retrieve(question, k=4):
    """Return the k most relevant chunks with similarity scores."""
    k = min(k, index.ntotal)
    q_vec = embedder.encode(["query: " + question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_vec, k)
    return [(chunks[i], float(s)) for i, s in zip(ids[0], scores[0])]

# Quick test
for chunk, score in retrieve("How can I get free bus travel?"):
    print(f"{score:.2f} | {chunk['title']}")


# Generation (Groq LLM Answers from Context)

In [ ]:
BILINGUAL_PROMPT = f"""You are a helpful assistant for Tamil Nadu government schemes.
Rules:
1. Answer ONLY using the CONTEXT given. Do not use outside knowledge.
2. If the context does not contain the answer, say you could not find it in the available documents and advise checking tn.gov.in or the concerned department.
3. Be short and clear. Use simple words.
4. ALWAYS give the answer in TWO parts, whatever language the question is in:
   English: <answer in English>
   தமிழ்: <the same answer in Tamil>
5. Mention the scheme name(s) you used.
6. If the context says an amount or change is "unconfirmed" or "not confirmed", say so. Never present it as confirmed.
7. End by advising the user to confirm details on the official portal.

Rules for the Tamil part:
- Use ONLY these exact Tamil scheme names. Never transliterate or invent a Tamil name:
{GLOSSARY}
- Write short, simple sentences in standard written Tamil.
- Keep numbers, ₹ amounts, dates, website names and phone numbers exactly as in the context.
- Use "உதவி எண்" for helpline.
- If you are unsure of a Tamil word, use the English word instead of guessing.
"""
SYSTEM_PROMPT = BILINGUAL_PROMPT   # one prompt for ask() and ask_bilingual()

def generate_answer(question, retrieved):
    context = "\n\n".join(
        f"[{n}] ({chunk['title']}) {chunk['text']}"
        for n, (chunk, _) in enumerate(retrieved, start=1)
    )
    user_message = f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.1,
    )
    return response.choices[0].message.content or "(The model returned an empty reply. Please try again.)"


# ask() Function (Full RAG Pipeline)

In [ ]:
def ask(question, k=4, show_sources=True):
    print("Q:", question)
    try:
        retrieved = retrieve(question, k)
        answer = generate_answer(question, retrieved)
    except Exception as e:
        print("\nSorry, something went wrong:", e)
        print("\n" + "=" * 70 + "\n")
        return
    print("\nA:", answer)
    if show_sources:
        print("\nSources used:")
        for chunk, score in retrieved:
            print(f"  - {chunk['title']} (similarity {score:.2f}) | {chunk['source']}")
    print("\n" + "=" * 70 + "\n")

print("ask() is ready")

# Demo Questions

In [ ]:
ask("My family income is Rs 1 lakh. Can I get free medical treatment? How much is covered?")
ask("Who is eligible for Kalaignar Magalir Urimai Thogai?")
ask("I am a girl who studied in a government school and joined college. Is there any monthly help?")
ask("Do my children get free breakfast in school?")
ask("What is the capital of France?")    # out of scope: should refuse politely
ask("மகளிர் உரிமைத் தொகை எவ்வளவு?")       # Tamil question

# Load Real PDFs (Optional)



In [ ]:
def load_pdf(path, title, source):
    reader = PdfReader(path)
    text = " ".join((page.extract_text() or "") for page in reader.pages)
    return {"title": title, "source": source, "tamil_name": "", "text": " ".join(text.split())}

pdf_files = glob.glob("/content/*.pdf")

if not pdf_files:
    print("No PDFs uploaded, so the sample data is kept. Nothing to do here.")
else:
    new_docs = []
    for p in pdf_files:
        name = os.path.basename(p).replace(".pdf", "")
        doc = load_pdf(p, name, os.path.basename(p))
        print(f"{name}: {len(doc['text'])} characters")   # very small number = scanned PDF, unreadable
        if len(doc["text"]) > 50:
            new_docs.append(doc)
    if new_docs:
        documents = new_docs
        chunks = build_chunks(documents)
        index, chunk_vectors = build_index(chunks)
        print(f"Now using {len(documents)} PDF(s) -> {len(chunks)} chunks. Ask questions with ask('...')")
    else:
        print("No readable text found in the PDFs. Sample data is kept.")

# Interactive Q&A Loop (English and Tamil Answers)

In [ ]:
generate_bilingual_answer = generate_answer

def ask_bilingual(question, k=4):
    print("\nQ:", question)
    try:
        retrieved = retrieve(question, k)
        answer = generate_bilingual_answer(question, retrieved)
    except Exception as e:
        print("\nSorry, something went wrong:", e)
        return
    print("\n" + answer)
    print("\nSources used:")
    for chunk, score in retrieved:
        print(f"  - {chunk['title']} (similarity {score:.2f}) | {chunk['source']}")
    print("\n" + "=" * 70)

# ---- The loop: keeps asking until you type exit ----
print("TN Scheme Assistant. Ask in English or Tamil. Type 'exit' to stop.\n")
while True:
    q = input("\nYour question (or 'exit'): ").strip()
    if q.lower() in ["exit", "quit", "bye"]:
        print("Thank you! Goodbye / நன்றி!")
        break
    if q == "":
        continue          # ignore empty input
    ask_bilingual(q)